# Thyroid Cancer Recurrence: Logistic Regression Classification

A logistic regression model for predicting differentiated thyroid cancer recurrence. The notebook:

- compares an unweighted model against a class-weighted model with 5-fold cross-validation
- tunes the regularization strength `C` and class weighting with a grid search
- evaluates the tuned model on a held-out test set, focusing on **recall** for the recurrence class
- reads the model's coefficients to see which factors drive recurrence

## Contents

1. Background
2. Setup and Constants
3. Load Data
4. Preprocessing
5. Model Pipeline
6. Baseline: Unweighted vs. Class-Weighted
7. Hyperparameter Tuning
8. Test Set Evaluation
9. Coefficients
10. Summary

## 1. Background

`LogisticRegression` models the probability that a patient recurred as a weighted sum of their features passed through the sigmoid function. A patient is predicted to recur when that probability is at least 0.5.

Each feature gets one coefficient. A positive coefficient raises the odds of recurrence and a negative one lowers it, which makes the model easy to interpret.

| Hyperparameter | Values tried | Effect |
|---|---|---|
| `C` | 0.01 to 100 | Inverse regularization strength. Smaller values shrink coefficients more and limit overfitting. |
| `class_weight` | `None`, `"balanced"` | `"balanced"` makes errors on the minority class (recurred) cost more, which targets recall. |
| `max_iter` | 1000 | Maximum solver iterations, set high enough to converge. |

### Why recall?

About 28% of patients recurred. A model that always predicts "no recurrence" would score about 72% accuracy while catching no recurrences at all. Recall measures the share of real recurrences the model catches, and a missed recurrence is the costly error.

## 2. Setup and Constants

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn import preprocessing

In [ ]:
TARGET_COL = "Recurred"
CATEGORICAL_COLS = [
    "Gender",
    "Smoking",
    "Hx Smoking",
    "Hx Radiotherapy",
    "Thyroid Function",
    "Physical Examination",
    "Adenopathy",
    "Pathology",
    "Focality",
    "Risk",
    "T",
    "N",
    "M",
    "Stage",
    "Response"
]
NUMERIC_COLS = ["Age"]

RAND_STATE = 42
TEST_SIZE = 0.25

# Metrics reported for every evaluation
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]

In [ ]:
# Formats a confusion matrix (cm) into a Pandas dataframe
def make_cm_frame(cm):
    return pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Pred No", "Pred Yes"]
    )


# Builds the full pipeline: encoding/scaling followed by logistic regression
def make_model(C=1.0, class_weight=None):
    return Pipeline([
        ("trans", colTransform),
        ("clf", LogisticRegression(
            C=C,
            class_weight=class_weight,
            max_iter=1000,
            random_state=RAND_STATE
        ))
    ])

### Cross-Validation Setup

`StratifiedKFold` keeps the ratio of target values consistent across each fold.

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAND_STATE)

## 3. Load Data

In [ ]:
df = pd.read_csv("../data/processed/Thyroid_Diff.csv")

print(df.shape)

df.head()

(383, 17)


,Age,Gender,Smoking,Hx Smoking,Hx Radiotherapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


## 4. Preprocessing

`X` holds the features to train on. `y` holds the target to predict. The target column maps Yes/No to 1/0.

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({"No": 0, "Yes": 1})

print(y.value_counts())
print(f"\nRecurrence rate: {y.mean():.1%}")

Recurred
0    275
1    108
Name: count, dtype: int64

Recurrence rate: 28.2%


### Encoding and Scaling

Columns in `CATEGORICAL_COLS` are one-hot encoded and `Age` is standardized to a mean of 0 and a standard deviation of 1. Scaling matters for logistic regression: regularization penalizes all coefficients equally, so features need to be on comparable scales.

Because the `ColumnTransformer` sits inside the pipeline, it is fit on training data only in every split and every cross-validation fold, so no information leaks from the test data.

In [ ]:
colTransform = ColumnTransformer(
    transformers=[
        ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ("num", preprocessing.StandardScaler(), NUMERIC_COLS)
    ],
    verbose_feature_names_out=False
)

## 5. Model Pipeline

In [ ]:
make_model()

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('trans', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{trans

## 6. Baseline: Unweighted vs. Class-Weighted

Both models use the default `C=1.0` and are scored with 5-fold cross-validation on the full dataset.

In [ ]:
baseline_models = {
    "Unweighted": make_model(),
    "Class-weighted (balanced)": make_model(class_weight="balanced"),
}

baseline_rows = []
baseline_cms = {}

for name, mdl in baseline_models.items():
    scores = cross_validate(mdl, X, y, cv=kfold, scoring=SCORING)
    row = {"Model": name}
    for metric in SCORING:
        row[metric] = scores[f"test_{metric}"].mean()
    baseline_rows.append(row)

    cv_pred = cross_val_predict(mdl, X, y, cv=kfold)
    baseline_cms[name] = confusion_matrix(y, cv_pred)

baseline_df = pd.DataFrame(baseline_rows)
print(baseline_df.to_string(index=False, float_format="{:.4f}".format))

                    Model  accuracy  precision  recall     f1  roc_auc
               Unweighted    0.9608     0.9628  0.8978 0.9278   0.9825
Class-weighted (balanced)    0.9451     0.8848  0.9255 0.9043   0.9803


In [ ]:
for name, cm in baseline_cms.items():
    print(f"Confusion matrix, 5-fold CV ({name}):")
    display(make_cm_frame(cm))

Confusion matrix, 5-fold CV (Unweighted):


,Pred No,Pred Yes
Actual No,271,4
Actual Yes,11,97


Confusion matrix, 5-fold CV (Class-weighted (balanced)):


,Pred No,Pred Yes
Actual No,262,13
Actual Yes,8,100


## 7. Hyperparameter Tuning

The data is split into a stratified training set and a held-out test set. `GridSearchCV` searches `C` and `class_weight` using 5-fold cross-validation on the **training set only**, so the test set stays untouched until the final evaluation.

The search selects the model with the best **F1** score. Selecting on recall alone can reward a model that flags nearly every patient as recurring; F1 balances recall against precision. Recall is still reported for every candidate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RAND_STATE
)

print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

Train: 287 rows, Test: 96 rows


In [ ]:
param_grid = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"],
}

search = GridSearchCV(
    make_model(),
    param_grid,
    cv=kfold,
    scoring=SCORING,
    refit="f1"
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print(f"Best CV F1: {search.best_score_:.4f}")

Best parameters: {'clf__C': 1, 'clf__class_weight': 'balanced'}
Best CV F1: 0.8935


In [ ]:
grid_df = pd.DataFrame(search.cv_results_)
grid_df = grid_df[[
    "param_clf__C",
    "param_clf__class_weight",
    "mean_test_recall",
    "mean_test_precision",
    "mean_test_f1",
    "mean_test_accuracy",
]].rename(columns={
    "param_clf__C": "C",
    "param_clf__class_weight": "class_weight",
    "mean_test_recall": "recall",
    "mean_test_precision": "precision",
    "mean_test_f1": "f1",
    "mean_test_accuracy": "accuracy",
})
grid_df["class_weight"] = grid_df["class_weight"].fillna("None")

print(grid_df.sort_values("f1", ascending=False).to_string(index=False, float_format="{:.4f}".format))

       C class_weight  recall  precision     f1  accuracy
  1.0000     balanced  0.9000     0.8987 0.8935    0.9376
  1.0000         None  0.8507     0.9382 0.8904    0.9410
 10.0000         None  0.8632     0.9129 0.8829    0.9341
 10.0000     balanced  0.8875     0.8870 0.8801    0.9306
  0.1000     balanced  0.8875     0.8349 0.8560    0.9165
100.0000     balanced  0.8382     0.8693 0.8479    0.9133
100.0000         None  0.8265     0.8748 0.8461    0.9133
  0.1000         None  0.7757     0.9300 0.8426    0.9199
  0.0100     balanced  0.8750     0.7803 0.8221    0.8955
  0.0100         None  0.4074     1.0000 0.5638    0.8329


## 8. Test Set Evaluation

The tuned model is scored once on the held-out test set.

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["No", "Yes"], digits=4))

              precision    recall  f1-score   support

          No     0.9714    0.9855    0.9784        69
         Yes     0.9615    0.9259    0.9434        27

    accuracy                         0.9688        96
   macro avg     0.9665    0.9557    0.9609        96
weighted avg     0.9686    0.9688    0.9686        96



In [ ]:
print("Confusion matrix (test set):")
display(make_cm_frame(confusion_matrix(y_test, y_pred)))

Confusion matrix (test set):


,Pred No,Pred Yes
Actual No,68,1
Actual Yes,2,25


## 9. Coefficients

Each coefficient is the change in the log-odds of recurrence for that feature. For one-hot columns, it compares patients in that category against the average effect of the column's other categories. Positive values push toward recurrence and negative values push away from it.

In [ ]:
feature_names = best_model.named_steps["trans"].get_feature_names_out()
coefs = best_model.named_steps["clf"].coef_[0]

coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coefs})
coef_df["Odds ratio"] = np.exp(coef_df["Coefficient"])

print("Top 10 features pushing TOWARD recurrence:")
print(coef_df.sort_values("Coefficient", ascending=False).head(10).to_string(index=False, float_format="{:.3f}".format))
print("\nTop 10 features pushing AWAY from recurrence:")
print(coef_df.sort_values("Coefficient").head(10).to_string(index=False, float_format="{:.3f}".format))

Top 10 features pushing TOWARD recurrence:
                                 Feature  Coefficient  Odds ratio
          Response_Structural Incomplete        2.706      14.970
         Response_Biochemical Incomplete        0.798       2.221
                               Risk_High        0.773       2.166
Thyroid Function_Clinical Hypothyroidism        0.729       2.073
                                   N_N1b        0.662       1.939
                                   T_T3b        0.620       1.858
                                Stage_II        0.605       1.832
                                   T_T1b        0.548       1.729
              Thyroid Function_Euthyroid        0.531       1.701
                    Pathology_Follicular        0.501       1.651

Top 10 features pushing AWAY from recurrence:
                                    Feature  Coefficient  Odds ratio
                         Response_Excellent       -2.365       0.094
                     Response_Indeterminate   

## 10. Summary

| Evaluation | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| 5-fold CV, unweighted (`C=1`) | 0.961 | 0.963 | 0.898 | 0.928 |
| 5-fold CV, class-weighted (`C=1`) | 0.945 | 0.885 | **0.926** | 0.904 |
| Tuned model on test set (`C=1`, balanced) | 0.969 | 0.962 | **0.926** | 0.943 |

**Findings**

- **Class weighting trades precision for recall.** Across 5-fold CV it catches 100 of 108 recurrences instead of 97, at the cost of 13 false alarms instead of 4. Since a missed recurrence is the costly error, the weighted model is the better fit for this problem.
- **Moderate regularization works best.** `C=1` came out on top. Very strong regularization (`C=0.01`, unweighted) collapses recall to about 41%, because the model barely predicts recurrence at all.
- **On the held-out test set**, the tuned model misses 2 of 27 recurrences and raises 1 false alarm.
- **Response to therapy dominates.** A *structural incomplete* response multiplies the odds of recurrence by about 15, while an *excellent* response cuts them to about a tenth. High risk category, N1b nodal spread, T3b tumors and stage II also push toward recurrence.

**Caveats**

- The test set has only 96 patients (27 recurrences), so one or two predictions shift the test metrics by several points. The cross-validation numbers are the more reliable comparison.
- Coefficients for rare categories (for example, Hurthel cell pathology or the less common thyroid function values) rest on very few patients and should not be over-interpreted.
- `Response` is recorded after treatment. A model trained on diagnosis-time features alone would show how much of this performance depends on it (see Future Work in the README).